In [1]:
import pandas as pd
from pathlib import Path

# Load both samples
base = Path("../data/processed")

medicare = pd.read_csv(
    base / "medicare_sample.csv",
    dtype={"Prscrbr_NPI": "string"}
)

payments = pd.read_csv(
    base / "payments_sample.csv",
    dtype={"covered_recipient_npi": "string"}
)

# Standardize NPI columns
medicare["npi"] = medicare["Prscrbr_NPI"].str.strip()
payments["npi"] = payments["covered_recipient_npi"].str.strip()

# Keep valid 10-digit NPIs
medicare = medicare[medicare["npi"].str.fullmatch(r"\d{10}").fillna(False)]
payments = payments[payments["npi"].str.fullmatch(r"\d{10}").fillna(False)]

# Find overlapping providers
medicare_npis = set(medicare["npi"])
payment_npis = set(payments["npi"])

matching_npis = medicare_npis & payment_npis

print("NPI JOIN VALIDATION")
print("=" * 45)
print("Medicare unique providers:", len(medicare_npis))
print("Open Payments unique providers:", len(payment_npis))
print("Matching providers:", len(matching_npis))
print("Example matches:", sorted(matching_npis)[:10])

NPI JOIN VALIDATION
Medicare unique providers: 2
Open Payments unique providers: 475
Matching providers: 0
Example matches: []


In [3]:
import requests

# Medicare Part D API
url = (
    "https://data.cms.gov/data-api/v1/dataset/"
    "9552739e-3d05-4c1b-8eff-ecabf391e2e5/data"
)

# Select 20 NPIs from Open Payments
test_npis = sorted(payment_npis)[:20]

results = []

for npi in test_npis:
    response = requests.get(
        url,
        params={
            "filter[Prscrbr_NPI]": npi,
            "size": 100
        },
        timeout=60
    )

    response.raise_for_status()
    records = response.json()

    if records:
        results.extend(records)

# Create DataFrame
matched_medicare = pd.DataFrame(results)

print("NPIs tested:", len(test_npis))
print("Medicare records found:", len(matched_medicare))

if not matched_medicare.empty:
    print(
        "Matching providers:",
        matched_medicare["Prscrbr_NPI"].nunique()
    )

    display(
        matched_medicare[
            ["Prscrbr_NPI", "Brnd_Name", "Tot_Clms"]
        ].head(10)
    )

NPIs tested: 20
Medicare records found: 233
Matching providers: 15


,Prscrbr_NPI,Brnd_Name,Tot_Clms
0,1003804964,Gabapentin,12
1,1003804964,Tramadol Hcl,48
2,1003901612,Brukinsa,14
3,1003901612,Imbruvica,16
4,1003901612,Levofloxacin,11
5,1003901612,Valacyclovir,30
6,1003901612,Venclexta,18
7,1003901612,Voriconazole,12
8,1003925991,Cephalexin,12
9,1013002328,Alprazolam,33


In [ ]:
# Count how many drug records we found for each provider
summary = (
    matched_medicare
    .groupby("Prscrbr_NPI")
    .size()
    .reset_index(name="Drug_Records")
    .sort_values("Drug_Records", ascending=False)
)

display(summary)

print("Total providers:", len(summary))
print("Total drug records:", summary["Drug_Records"].sum())

,Prscrbr_NPI,Drug_Records
3,1013002328,90
9,1023221959,46
6,1013902444,23
11,1033170154,20
5,1013414135,8
13,1033409131,7
14,1033451950,7
1,1003901612,6
8,1023011897,6
12,1033175484,6


Total providers: 15
Total drug records: 233


# Aggregate NPI's claim and amount

In [13]:
# Convert columns to numeric
matched_medicare["Tot_Clms"] = pd.to_numeric(
    matched_medicare["Tot_Clms"],
    errors="coerce"
)

matched_medicare["Tot_Drug_Cst"] = pd.to_numeric(
    matched_medicare["Tot_Drug_Cst"],
    errors="coerce"
)

# Aggregate by NPI
summary = (
    matched_medicare
    .groupby("Prscrbr_NPI")
    .agg(
        Drug_Records=("Brnd_Name", "size"),
        Total_Claims=("Tot_Clms", "sum"),
        Total_Drug_Cost=("Tot_Drug_Cst", "sum")
    )
    .reset_index()
    .sort_values("Total_Claims", ascending=False)
)

display(summary)

print("Total providers:", len(summary))
print("Total drug records:", summary["Drug_Records"].sum())
print("Total claims:", summary["Total_Claims"].sum())
print("Total drug cost:", (summary["Total_Drug_Cost"].sum()))



,Prscrbr_NPI,Drug_Records,Total_Claims,Total_Drug_Cost
3,1013002328,90,5728,2742063.19
9,1023221959,46,3168,1608086.05
11,1033170154,20,2515,225202.52
6,1013902444,23,611,351586.04
12,1033175484,6,267,30349.45
8,1023011897,6,211,1812.50
10,1023636057,5,172,820.23
14,1033451950,7,156,898.50
13,1033409131,7,150,1075.31
5,1013414135,8,147,20781.96


Total providers: 15
Total drug records: 233
Total claims: 13457
Total drug cost: 5463636.549999998


# Aggregate Open Payments by NPI

In [14]:
# Clean the provider identifiers
payments["npi"] = (
    payments["covered_recipient_npi"]
    .astype("string")
    .str.strip()
)

# Convert payment amounts to numeric
payments["total_amount_of_payment_usdollars"] = pd.to_numeric(
    payments["total_amount_of_payment_usdollars"],
    errors="coerce"
)

# Keep valid NPIs
payments_clean = payments[
    payments["npi"].str.fullmatch(r"\d{10}").fillna(False)
].copy()

# Aggregate payments by NPI
payments_agg = (
    payments_clean
    .groupby("npi", as_index=False)
    .agg(
        Payment_Records=("total_amount_of_payment_usdollars", "size"),
        Total_Payments=("total_amount_of_payment_usdollars", "sum")
    )
    .sort_values("Total_Payments", ascending=False)
)

display(payments_agg.head(10))

print("Original payment records:", len(payments))
print("Unique providers:", len(payments_agg))
print("Total payments:", payments_agg["Total_Payments"].sum())

,npi,Payment_Records,Total_Payments
410,1821096207,1,30000.00
206,1467488841,1,22475.51
104,1235137878,1,11400.00
122,1265813992,1,5800.00
13,1023221959,1,5330.00
110,1245309103,1,5220.00
452,1942302138,1,5166.48
414,1841216157,1,5050.00
425,1871538116,1,4830.00
91,1194782771,1,4800.00


Original payment records: 494
Unique providers: 475
Total payments: 168172.03


# Both Joined after Aggregation

In [15]:
# Prepare Medicare NPI for joining
summary["npi"] = (
    summary["Prscrbr_NPI"]
    .astype("string")
    .str.strip()
)

# Join both aggregated datasets on NPI
joined = pd.merge(
    summary,
    payments_agg,
    on="npi",
    how="inner",
    validate="one_to_one"
)

# Display results
print("JOIN RESULTS")
print("=" * 50)

print("Medicare providers:", len(summary))
print("Open Payments providers:", len(payments_agg))
print("Successfully joined providers:", len(joined))

print("\nJOINED DATA:")
display(
    joined[
        [
            "npi",
            "Drug_Records",
            "Total_Claims",
            "Total_Drug_Cost",
            "Payment_Records",
            "Total_Payments"
        ]
    ].sort_values("Total_Payments", ascending=False)
)

JOIN RESULTS
Medicare providers: 15
Open Payments providers: 475
Successfully joined providers: 15

JOINED DATA:


,npi,Drug_Records,Total_Claims,Total_Drug_Cost,Payment_Records,Total_Payments
1,1023221959,46,3168,1608086.05,1,5330.00
6,1023636057,5,172,820.23,1,746.68
0,1013002328,90,5728,2742063.19,1,355.22
12,1003804964,2,60,120.13,1,128.17
8,1033409131,7,150,1075.31,1,91.50
7,1033451950,7,156,898.50,1,38.39
5,1023011897,6,211,1812.50,1,37.53
13,1013334291,2,39,92.85,1,24.68
11,1003901612,6,101,479748.49,1,20.00
4,1033175484,6,267,30349.45,1,18.88


In [20]:
matched_medicare.to_csv(
    "../data/processed/joined_medicare_sample.csv",
    index=False
)